In [1]:
%matplotlib inline
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance
from scipy import stats
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [2]:
"""
You can start the code from here
"""
#Import the actual data
df_biofilm_filtered = pd.read_csv('Figurewise raw data/Biofilm_filtered.csv', index_col = 0)
df_biofilm_filtered.head()

,3,8,28,32,42,64,70,78,100,108,...,t_27_enrichment,t_39_enrichment,t_45_enrichment,t_63_enrichment,t_73_enrichment,t_81_enrichment,t_99_enrichment,t_107_enrichment,t_115_enrichment,t_137_enrichment
barcode,,,,,,,,,,,,,,,,,,,,,
GCGTATCGGAACTAGAAGGG,0,1,7,3,1,0,3,1,3,3,...,0.301030,0.602060,0.301030,0.602060,1.255273,1.146128,0.903090,0.000000,0.000000,0.000000
ACTGGAGAGACGCTCCAAGT,0,5,1,0,2,0,1,2,0,0,...,0.301030,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.301030,0.000000
GTTTTTAAATGTGTGCCCGG,1,4,1,3,9,0,4,0,0,0,...,-0.301030,-0.301030,0.397940,0.301030,-0.301030,-0.301030,-0.301030,-0.301030,-0.301030,-0.301030
GTACTAAATTGATGCCCATT,10,0,0,0,0,0,0,2,3,0,...,-0.740363,-1.041393,-1.041393,-0.740363,-1.041393,-1.041393,-1.041393,-1.041393,-1.041393,-1.041393
GGCCAAAGTACCTGTTAAAT,7,4,4,4,6,10,5,9,3,4,...,-0.903090,-0.425969,0.096910,-0.903090,-0.903090,-0.425969,-0.903090,-0.903090,-0.425969,-0.903090


In [3]:
#We extract neutral barcodes. The neutral barcodes were determined using sequencing and correspond to barcodes without mutations.
Neutral_barcodes = pd.read_csv('Neutral_barcodes.csv', index_col=0)
Neutral_barcodes_true = []

#Extract the true barcodes from the neutral barcode list.
for barcode in Neutral_barcodes.Barcode:
    Neutral_barcodes_true.append(barcode[2:7]+barcode[9:14]+barcode[16:21]+barcode[23:28])

#We make a dataframe with just the Neutral barcodes.
df_neutral = df_biofilm_filtered.loc[df_biofilm_filtered.index.isin(Neutral_barcodes_true)]

#We create an artificial barcode called as 'WT', which is the sum of all neutral barcode.
#This wildtype barcode serves as a reference for all neutral barcodes.
df_biofilm_filtered.loc['WT'] = df_neutral.sum()
df_biofilm_filtered

,3,8,28,32,42,64,70,78,100,108,...,t_27_enrichment,t_39_enrichment,t_45_enrichment,t_63_enrichment,t_73_enrichment,t_81_enrichment,t_99_enrichment,t_107_enrichment,t_115_enrichment,t_137_enrichment
barcode,,,,,,,,,,,,,,,,,,,,,
GCGTATCGGAACTAGAAGGG,0.0,1.0,7.0,3.0,1.0,0.0,3.0,1.0,3.0,3.0,...,0.301030,0.602060,0.301030,0.602060,1.255273,1.146128,0.903090,0.000000,0.000000,0.000000
ACTGGAGAGACGCTCCAAGT,0.0,5.0,1.0,0.0,2.0,0.0,1.0,2.0,0.0,0.0,...,0.301030,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.301030,0.000000
GTTTTTAAATGTGTGCCCGG,1.0,4.0,1.0,3.0,9.0,0.0,4.0,0.0,0.0,0.0,...,-0.301030,-0.301030,0.397940,0.301030,-0.301030,-0.301030,-0.301030,-0.301030,-0.301030,-0.301030
GTACTAAATTGATGCCCATT,10.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,...,-0.740363,-1.041393,-1.041393,-0.740363,-1.041393,-1.041393,-1.041393,-1.041393,-1.041393,-1.041393
GGCCAAAGTACCTGTTAAAT,7.0,4.0,4.0,4.0,6.0,10.0,5.0,9.0,3.0,4.0,...,-0.903090,-0.425969,0.096910,-0.903090,-0.903090,-0.425969,-0.903090,-0.903090,-0.425969,-0.903090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CTTGACGGAGGGCTG-ATTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
CGCACAAGAGAACGCGAAAC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.477121
CAGTAATTTCGTTATATAAT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [7]:
W0 = ['3', '9', '29']
W1 = ['9', '29', '46']
W2 = ['29','46','65']
W3 = ['46', '65', '74']
W4 = ['65', '74', '82']
W5 = ['74', '82', '101']
W6 = ['82', '101', '109']
W7 = ['101', '109', '117']
W8 = ['109', '117', '138']
W9 = ['117', '138', '142']
W10 = ['138', '142', '146']
W11 = ['142', '146', '161']
W12 = ['146', '161', '165']
W13 = ['161', '165', '169']

list_windows = [W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,W10,W11,W12,W13]

i = 0
window_names = ['A','B','C','D','E','F','G','H','I','J','K','L','M','N']
for window in list_windows:
    print(window, window_names[i])
    #df_pop_DMS = df_biofilm_filtered[window].rename(columns={window[0]:'0',window[1]:'6.64',window[2]:'13.28'})
    df_pop_DMS = df_biofilm_filtered[window].rename(columns={window[0]:'0',window[1]:'1',window[2]:'2'})
    df_pop_DMS['is_WT'] = 'False'
    df_pop_DMS.loc['WT','is_WT'] = 'True'
    #if window == ['W3']:
    #    df_pop_DMS.drop('19.92', axis = 1)
    df_pop_DMS_final = pd.DataFrame(df_pop_DMS.reset_index().set_index(['barcode','is_WT']).stack()).reset_index().rename(columns = {'level_2':'generation',0:'counts'})
    df_pop_DMS_final.to_csv('popDMS/PopDMSA_Biofilms_W'+window_names[i]+'.csv')
    i = i+1

['3', '9', '29'] A
['9', '29', '46'] B
['29', '46', '65'] C
['46', '65', '74'] D
['65', '74', '82'] E
['74', '82', '101'] F
['82', '101', '109'] G
['101', '109', '117'] H
['109', '117', '138'] I
['117', '138', '142'] J
['138', '142', '146'] K
['142', '146', '161'] L
['146', '161', '165'] M
['161', '165', '169'] N


In [ ]:
window = ['9', '29', '46']
#df_pop_DMS = df_biofilm_filtered[window].rename(columns={window[0]:'0',window[1]:'6.64',window[2]:'19.93'})
df_pop_DMS = df_biofilm_filtered[window].rename(columns={window[0]:'0',window[1]:'1',window[2]:'3'})
df_pop_DMS['is_WT'] = 'False'
df_pop_DMS.loc['WT','is_WT'] = 'True'
    
df_pop_DMS_final = pd.DataFrame(df_pop_DMS.reset_index().set_index(['barcode','is_WT']).stack()).reset_index().rename(columns = {'level_2':'generation',0:'counts'})
df_pop_DMS_final.to_csv('popDMS/PopDMSA_Biofilms_WB.csv')

window = ['29', '46', '65']
#df_pop_DMS = df_biofilm_filtered[window].rename(columns={window[0]:'0',window[1]:'13.28',window[2]:'19.93'})
df_pop_DMS = df_biofilm_filtered[window].rename(columns={window[0]:'0',window[1]:'2',window[2]:'3'})
df_pop_DMS['is_WT'] = 'False'
df_pop_DMS.loc['WT','is_WT'] = 'True'
    
df_pop_DMS_final = pd.DataFrame(df_pop_DMS.reset_index().set_index(['barcode','is_WT']).stack()).reset_index().rename(columns = {'level_2':'generation',0:'counts'})
df_pop_DMS_final.to_csv('popDMS/PopDMSA_Biofilms_WC.csv')